# Demo 07. Floating-Point Arithmetic and Catastrophic Cancellation

**Standard 6** (conditioning and stability), Week 5.

Floating-point numbers carry finite precision. Subtracting two nearly equal quantities cancels their leading digits and leaves a result dominated by rounding error. This demo measures the effect against a 50-digit reference and shows that an algebraically equivalent formula removes it.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
from ipywidgets import interact, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)
mp.mp.dps = 50   # 50 decimal digits, used as the "exact" reference

## The source of the error

IEEE double precision represents a real number with a relative error of at most the unit roundoff $u \approx 1.1\times10^{-16}$. If $a$ and $b$ agree to $k$ leading digits, their stored forms already carry absolute error near $u\lvert a\rvert$, and the difference $a-b$ retains that absolute error while its own magnitude has dropped by a factor of $10^{k}$. The relative error is therefore amplified by roughly $10^{k}$. The remedy is to rewrite the expression so the subtraction of nearly equal quantities does not occur, for example

$$ 1 - \cos x = 2\sin^2\!\tfrac{x}{2}. $$

In [ ]:
# ---------------------------------------------------------------------------
# Machine epsilon and the limits of float64
# ---------------------------------------------------------------------------
# IEEE double precision stores about 15 to 16 significant decimal digits.
# Machine epsilon is the gap between 1.0 and the next representable number.
# Subtracting two nearly equal numbers cancels leading digits and exposes the
# rounding error left in the trailing ones. This is catastrophic cancellation.

print("float64 machine epsilon :", np.finfo(float).eps)
print("1.0 + eps/2 == 1.0      :", 1.0 + np.finfo(float).eps / 2 == 1.0)
print("0.1 + 0.2 - 0.3         :", 0.1 + 0.2 - 0.3)   # not exactly zero

In [ ]:
# ---------------------------------------------------------------------------
# A worked example: f(x) = (1 - cos x) / x^2, with limit 1/2 as x -> 0
# ---------------------------------------------------------------------------
# Near x = 0, cos x is close to 1, so 1 - cos x cancels almost all significant
# digits before the division. An algebraically equivalent form avoids the
# subtraction and stays accurate:
#
#     1 - cos x = 2 sin^2(x/2),   so   f(x) = 2 sin^2(x/2) / x^2.

def f_naive(x):
    """Direct evaluation of (1 - cos x)/x^2. Loses accuracy for small x."""
    return (1.0 - np.cos(x)) / x**2

def f_stable(x):
    """Algebraically equivalent form 2 sin^2(x/2)/x^2, free of cancellation."""
    s = np.sin(x / 2.0)
    return 2.0 * s * s / x**2

def f_exact(x):
    """Reference value at 50-digit precision using mpmath."""
    xx = mp.mpf(x)
    return float((1 - mp.cos(xx)) / xx**2)

In [ ]:
# ---------------------------------------------------------------------------
# Relative error of each formula against the high-precision reference
# ---------------------------------------------------------------------------
def show_cancellation(log10_x=-7.0):
    """Evaluate both formulas at x = 10**log10_x and report relative error
    against the 50-digit mpmath value."""
    x = 10.0 ** log10_x
    exact = f_exact(x)
    naive = f_naive(x)
    stable = f_stable(x)

    rel = lambda v: abs(v - exact) / abs(exact)
    print(f"x = {x:.1e}")
    print(f"  exact  (50 digits) : {exact:.16f}")
    print(f"  naive  formula     : {naive:.16f}   rel err = {rel(naive):.2e}")
    print(f"  stable formula     : {stable:.16f}   rel err = {rel(stable):.2e}")

def error_curve():
    """Plot relative error of both formulas across a range of x."""
    xs = np.logspace(-9, 0, 40)
    e_naive = [abs(f_naive(x) - f_exact(x)) / abs(f_exact(x)) for x in xs]
    e_stable = [abs(f_stable(x) - f_exact(x)) / abs(f_exact(x)) for x in xs]
    plt.figure()
    plt.loglog(xs, e_naive, "r.-", label="naive (1-cos x)/x^2")
    plt.loglog(xs, e_stable, "g.-", label="stable 2 sin^2(x/2)/x^2")
    plt.axhline(np.finfo(float).eps, color="k", ls=":", label="machine eps")
    plt.gca().invert_xaxis()
    plt.xlabel("x"); plt.ylabel("relative error"); plt.legend()
    plt.title("Catastrophic cancellation as x -> 0")
    plt.show()

show_cancellation(-7.0)
error_curve()

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: move x toward zero and compare the two formulas
# ---------------------------------------------------------------------------
interact(
    show_cancellation,
    log10_x=FloatSlider(value=-7.0, min=-9.0, max=-0.5, step=0.5, description="log10(x)"),
);

## Summary

- Double precision carries a relative error near $10^{-16}$ per operation.
- Subtracting nearly equal numbers amplifies relative error in proportion to the number of cancelled digits.
- An algebraically equivalent formula that avoids the subtraction preserves accuracy across the whole range of $x$.
- The error is a property of the formula and the arithmetic, distinct from the conditioning of the underlying problem (demo 06).